In [28]:
import os
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
import torch.optim as optim
from PIL import Image
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from torch.utils.data import random_split
import torch.nn as nn


In [29]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                        [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [30]:
if torch.backends.mps.is_available():
    device = "mps"   
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(device)

cuda


In [31]:
if os.name == "nt":
    ds_path = r"C:\Users\PAARTH\Documents\plant-disease-detector\data\plantvillage_dataset\plantvillage dataset\color"
else:
    ds_path = r"/Users/chandu/dev/peri-peri-beans/plant-disease-detector/data/plantvillage dataset/color"
ds_full = datasets.ImageFolder(ds_path)

num_classes = len(ds_full.classes)
class_names = ds_full.classes

print("Classes:",num_classes)

Classes: 38


In [32]:
train_size = int(0.8*(len(ds_full)))
val_size = len(ds_full) - train_size

train_dataset, val_dataset = random_split(ds_full, [train_size,val_size])

train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform = val_transform

In [33]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [38]:
class PlantCNN(nn.Module):
    def __init__(self, nn_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.AdaptiveAvgPool2d((1,1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, nn_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
        

In [39]:
#MODEL
model = PlantCNN(num_classes).to(device)

In [40]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(),lr = 0.001)